# S3.21 - badRecordsPath
**Date completed:** November 2026  
**Status:** Completed   
**Key concept:** Bad records are written to a separate path, with the reason, so they can be reviewed

In [0]:
# ============================================================
# S3.21 — Cell 2
# badRecordsPath: Capture Malformed CSV Records to Files
#
# Goal:
# 1. Read malformed CSV data using badRecordsPath.
# 2. See which rows remain in the returned DataFrame.
# 3. Inspect files created under badRecordsPath.
# 4. Inspect the captured bad record and failure reason.
# 5. Compare badRecordsPath with _corrupt_record from S3.20.
# 6. Understand production limitations of badRecordsPath.
#
# Dataset:
# Controlled 5-row CSV in a Unity Catalog Volume.
#
# Expected:
#
# Source rows        = 5
# Returned rows      = 4
#
# Row 3:
# 3,Pune,abc
# → malformed because qty expects IntegerType
# → captured under badRecordsPath
# → not returned in the DataFrame
#
# Row 4:
# 4,Chennai,40,EXTRA
# → expected to remain
# → EXTRA is outside the supplied schema
#
# Important:
#
# badRecordsPath takes precedence over _corrupt_record.
# A malformed record written to badRecordsPath does not
# appear in the returned DataFrame.
# ============================================================

from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType
)


print("=== S3.21: badRecordsPath ===")
print(f"Spark version: {spark.version}")
print()


# ------------------------------------------------------------
# Step 1 — Create Volume and Paths
# ------------------------------------------------------------

spark.sql("""
CREATE VOLUME IF NOT EXISTS workspace.default.s3_21_files
""")

BASE_PATH = (
    "/Volumes/workspace/default/"
    "s3_21_files"
)

CSV_PATH = (
    f"{BASE_PATH}/orders.csv"
)

BAD_PATH = (
    f"{BASE_PATH}/bad_records"
)

# Remove previous bad-record output so this run is isolated.

try:
    dbutils.fs.rm(
        BAD_PATH,
        True
    )
except Exception:
    pass

print(f"CSV path        : {CSV_PATH}")
print(f"Bad-record path : {BAD_PATH}")
print()

In [0]:
# ------------------------------------------------------------
# Step 2 — Create Controlled CSV
#
# Row 1 → valid
# Row 2 → valid
#
# Row 3:
# qty = abc
# → invalid IntegerType value
#
# Row 4:
# extra token
#
# Row 5 → valid
# ------------------------------------------------------------

lines = [

    "id,city,qty",

    "1,Delhi,10",

    "2,Mumbai,20",

    "3,Pune,abc",

    "4,Chennai,40,EXTRA",

    "5,Delhi,50",
]


source_rows = (
    len(lines) - 1
)

dbutils.fs.put(
    CSV_PATH,
    "\n".join(lines),
    True
)


print(
    f"Source data rows: {source_rows}"
)

print()

In [0]:
# ------------------------------------------------------------
# Step 3 — Explicit Schema
# ------------------------------------------------------------

schema = StructType([

    StructField(
        "id",
        IntegerType(),
        True
    ),

    StructField(
        "city",
        StringType(),
        True
    ),

    StructField(
        "qty",
        IntegerType(),
        True
    )
])

In [0]:
# ------------------------------------------------------------
# Step 4 — Prediction Before Execution
#
# Unlike S3.20:
#
# We are NOT keeping malformed rows in _corrupt_record.
#
# badRecordsPath stores the malformed record externally.
#
# Expected:
#
# 5 source rows
# - 1 malformed row
# = 4 returned rows
# ------------------------------------------------------------

print(
    "=== STEP 4: PREDICTION ==="
)


predicted_returned_rows = 4


print(
    f"Source rows             : {source_rows}"
)

print(
    f"Predicted returned rows : "
    f"{predicted_returned_rows}"
)

print(
    "Predicted bad record    : 3,Pune,abc"
)

print()




In [0]:
# ------------------------------------------------------------
# Step 5 — Read Using badRecordsPath
#
# Explicitly set PERMISSIVE so the lesson does not depend
# on any implicit/default parser mode.
#
# collect() is intentional here:
#
# - only 5 controlled source rows
# - forces parsing
# - lets us inspect every returned record
# ------------------------------------------------------------

print(
    "=== STEP 5: READ WITH badRecordsPath ==="
)


read_succeeded = False
returned_rows = []


try:

    df_bad_path = (

        spark.read

        .option(
            "header",
            "true"
        )

        .option(
            "mode",
            "PERMISSIVE"
        )

        .option(
            "badRecordsPath",
            BAD_PATH
        )

        .schema(
            schema
        )

        .csv(
            CSV_PATH
        )
    )

    returned_rows = (
        df_bad_path.collect()
    )


    read_succeeded = True


    print(
        f"Rows returned: "
        f"{len(returned_rows)}"
    )


    for row in returned_rows:

        print(row)


except Exception as e:

    print(
        "Read failed."
    )

    print(
        f"Exception type: "
        f"{type(e).__name__}"
    )

    print(
        f"Message preview: "
        f"{str(e)[:400]}"
    )


print()

In [0]:
# ------------------------------------------------------------
# Step 6 — Inspect Returned IDs
#
# If the prediction is correct:
#
# Returned IDs:
#
# 1
# 2
# 4
# 5
#
# ID 3 should not appear because it was redirected to
# badRecordsPath.
# ------------------------------------------------------------

if read_succeeded:

    print(
        "=== STEP 6: RETURNED ROWS ==="
    )


    returned_ids = sorted(
        row["id"]
        for row in returned_rows
    )


    print(
        f"Returned IDs: {returned_ids}"
    )


    print(
        f"Returned row count: "
        f"{len(returned_rows)}"
    )


    print()

In [0]:
# ------------------------------------------------------------
# Step 7 — Validate Returned Rows
#
# Do not validate the number of physical bad-record files.
#
# Spark controls physical file layout.
#
# Validate business/data behaviour instead.
# ------------------------------------------------------------

if read_succeeded:

    print(
        "=== STEP 7: DATAFRAME VALIDATION ==="
    )


    assert source_rows == 5, (
        "Unexpected source row count"
    )


    assert len(returned_rows) == 4, (
        "Expected 4 returned rows"
    )


    assert 3 not in returned_ids, (
        "Malformed row ID 3 should not "
        "be in returned DataFrame"
    )


    assert 4 in returned_ids, (
        "Row ID 4 was expected to remain"
    )


    print(
        "Returned-row validation passed."
    )

    print()

In [0]:
# ------------------------------------------------------------
# Step 8 — Recursively List badRecordsPath
#
# IMPORTANT:
#
# Do NOT silently swallow listing errors.
#
# Otherwise:
#
# "Files found = 0"
#
# could mean either:
#
# A. no bad record was written
#
# OR
#
# B. listing failed
#
# We need to distinguish those situations.
# ------------------------------------------------------------

def list_files_recursive(path):

    found = []

    items = dbutils.fs.ls(
        path
    )


    for item in items:

        if item.path.endswith("/"):

            found.extend(
                list_files_recursive(
                    item.path
                )
            )

        else:

            found.append(
                item.path
            )


    return found


print(
    "=== STEP 8: FILES IN badRecordsPath ==="
)


bad_files = []


try:

    bad_files = (
        list_files_recursive(
            BAD_PATH
        )
    )


    print(
        f"Physical files found: "
        f"{len(bad_files)}"
    )


    for file_path in bad_files:

        print(
            file_path
        )


except Exception as e:

    print(
        "Unable to list badRecordsPath."
    )

    print(
        f"Exception type: "
        f"{type(e).__name__}"
    )

    print(
        f"Message preview: "
        f"{str(e)[:400]}"
    )


print()

In [0]:
# ------------------------------------------------------------
# Step 9 — Read Bad-Record Files as Raw Text
#
# badRecordsPath stores diagnostic information about
# malformed records.
#
# We first inspect the files as TEXT so we do not assume
# a particular internal file schema.
# ------------------------------------------------------------

print(
    "=== STEP 9: BAD-RECORD FILE CONTENT ==="
)


bad_file_lines = []


for file_path in bad_files:

    # Ignore common metadata/control files if present.
    file_name = (
        file_path
        .rstrip("/")
        .split("/")[-1]
    )


    if (
        file_name.startswith("_")
        or
        file_name.startswith(".")
    ):

        continue


    try:

        text_rows = (

            spark.read

            .text(
                file_path
            )

            .collect()
        )


        for row in text_rows:

            value = (
                row["value"]
            )

            bad_file_lines.append(
                value
            )

            print(
                value[:500]
            )


    except Exception as e:

        print(
            f"Could not read: {file_path}"
        )

        print(
            f"{type(e).__name__}: "
            f"{str(e)[:200]}"
        )


print()




In [0]:
# ------------------------------------------------------------
# Step 10 — Validate Captured Bad Record
#
# We care about the DATA captured, not how many physical
# files Spark happened to create.
# ------------------------------------------------------------

print(
    "=== STEP 10: BAD-RECORD VALIDATION ==="
)


if read_succeeded:

    captured_text = (
        "\n".join(
            bad_file_lines
        )
    )


    captured_row_3 = (
        "3,Pune,abc"
        in
        captured_text
    )


    print(
        f"Captured '3,Pune,abc': "
        f"{captured_row_3}"
    )


    if bad_file_lines:

        assert captured_row_3, (
            "Expected malformed source row "
            "was not found in badRecordsPath"
        )


        print(
            "Bad-record validation passed."
        )

    else:

        print(
            "No readable bad-record content found. "
            "Inspect the actual path/output before "
            "drawing a conclusion."
        )


print()

In [0]:
# ------------------------------------------------------------
# Step 11 — Reconciliation
#
# Expected conceptual reconciliation:
#
# Source rows
# =
# returned good/tolerated rows
# +
# malformed records redirected externally
#
# For this exercise:
#
# 5
# =
# 4
# +
# 1 malformed record
# ------------------------------------------------------------

print(
    "=== STEP 11: RECONCILIATION ==="
)

if read_succeeded:

    print(
        f"Source rows       : {source_rows}"
    )

    print(
        f"Returned rows     : "
        f"{len(returned_rows)}"
    )

    print(
        f"Rows not returned : "
        f"{source_rows - len(returned_rows)}"
    )


    print("""
Expected interpretation:

5 source rows
      ↓
4 returned rows
+
1 malformed record redirected to badRecordsPath
""")

print()


In [0]:
# ------------------------------------------------------------
# Step 12 — S3.20 vs S3.21
# ------------------------------------------------------------

print(
    "=== STEP 12: S3.20 vs S3.21 ==="
)


print("""
S3.20
------------------------------------------------

PERMISSIVE + _corrupt_record

Malformed row
      ↓
remains in DataFrame
      ↓
bad parsed field = NULL
      ↓
raw source line available in _corrupt_record


S3.21
------------------------------------------------

PERMISSIVE + badRecordsPath

Malformed row
      ↓
written externally
      ↓
does NOT appear in returned DataFrame


Important:

badRecordsPath takes precedence over _corrupt_record
when both mechanisms are involved.
""")

In [0]:
# ------------------------------------------------------------
# Step 13 — Production Caveat
#
# badRecordsPath is useful for learning and some file-based
# ingestion scenarios.
#
# However, Databricks documents important limitations:
#
# - file-based badRecordsPath is non-transactional
# - inconsistent results are possible
#
# Databricks also recommends _corrupt_record rather than
# badRecordsPath for Auto Loader because of potential
# race-condition issues.
#
# Therefore:
#
# do not automatically choose badRecordsPath as the
# enterprise default just because it exists.
# ------------------------------------------------------------

print(
    "=== STEP 13: PRODUCTION CONSIDERATION ==="
)


print("""
badRecordsPath:

Useful for:
→ external bad-record capture
→ investigation
→ learning parser behaviour


But:

→ it is file based
→ it is non-transactional
→ production design should consider more robust
  DQ/quarantine approaches


For modern ingestion architectures, options such as:

_corrupt_record
_rescued_data
Auto Loader
Lakeflow expectations
explicit quarantine tables

may be more appropriate depending on the use case.
""")

In [0]:
# ------------------------------------------------------------
# Step 14 — Final Summary
# ------------------------------------------------------------

print(
    "=== FINAL LEARNING SUMMARY ==="
)


print("""
badRecordsPath:

External path used to capture information about
malformed CSV records.


In this exercise:

Source
→ 5 rows

Malformed:
→ 3,Pune,abc

Expected returned DataFrame:
→ 4 rows

Expected bad-record location:
→ BAD_PATH


Difference from _corrupt_record:

_corrupt_record
→ bad record remains visible in DataFrame

badRecordsPath
→ bad record is redirected externally
→ bad record does not appear in returned DataFrame


Important:

Do not validate physical bad-file count.

Validate:

→ returned rows
→ missing malformed row
→ captured bad-record content


Production principle:

Capture
      ↓
Inspect
      ↓
Classify
      ↓
Quarantine / remediate
      ↓
Reconcile
      ↓
Monitor
""")


print("=== S3.21 CELL 2 COMPLETED ===")

In [0]:
# ============================================================
# S3.21 - Cell 3: compact checkpoint (prints RESULT lines only)
#
# Goal:
# 1. Did the read with badRecordsPath succeed?
# 2. How many rows came back?
# 3. How many files landed in the bad-records path?
# 4. What does a bad-record file contain?
# ============================================================
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

base = "/Volumes/workspace/default/s3_21_files"
CSV_PATH = f"{base}/orders.csv"
BAD_PATH = f"{base}/bad_records_check"
dbutils.fs.rm(BAD_PATH, True)

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("city", StringType(), True),
    StructField("qty", IntegerType(), True),
])

# ------------------------------------------------------------
# Step 1 - Read inside try (parsing happens at the action)
# ------------------------------------------------------------
try:
    rows = (spark.read.option("header", "true")
            .option("badRecordsPath", BAD_PATH)
            .schema(schema).csv(CSV_PATH).collect())
    print("RESULT 1 read       : OK")
    print("RESULT 2 rows       :", [(r["id"], r["city"], r["qty"]) for r in rows])
except Exception as e:
    print("RESULT 1 read       : ERROR", type(e).__name__)
    print("RESULT 2 first line :", str(e).split("\n")[0][:200])

# ------------------------------------------------------------
# Step 2 - Files in the bad-records path
# ------------------------------------------------------------
def list_files(p):
    out = []
    try:
        for i in dbutils.fs.ls(p):
            out += list_files(i.path) if i.path.endswith("/") else [i.path]
    except Exception:
        pass
    return out

files = [f for f in list_files(BAD_PATH) if "/_" not in f]
print("RESULT 3 bad files  :", len(files))
for f in files[:2]:
    print("RESULT 4 content    :", spark.read.text(f).first()["value"][:250])

# S3.21 — Key Takeaways: badRecordsPath

## 1. What badRecordsPath Does

`badRecordsPath` stores information about malformed input records in a separate path so they can be investigated later.

Conceptually:

```text
Source CSV
    ↓
Parser
    ↓
┌─────────────────┬──────────────────────┐
│                 │                      │
Valid records     Malformed records
│                 │
Returned          badRecordsPath
DataFrame         files
```

When `badRecordsPath` captures a malformed record, that record does not appear in the returned DataFrame.

---

## 2. Comparison

| Option | Malformed Row | Trace / Evidence |
|---|---|---|
| `DROPMALFORMED` | Dropped | No retained bad-row payload |
| `_corrupt_record` | Kept in DataFrame | Raw malformed line stored in a column |
| `badRecordsPath` | Redirected outside DataFrame | Diagnostic record stored under the bad-record path |

### Important

When `badRecordsPath` is used, it takes precedence over `_corrupt_record`.

Therefore:

```text
_corrupt_record
→ malformed row remains in DataFrame

badRecordsPath
→ malformed row is written externally
→ malformed row is not returned
```

---

## 3. Expected S3.21 Behaviour

Controlled source:

```text
1,Delhi,10
2,Mumbai,20
3,Pune,abc
4,Chennai,40,EXTRA
5,Delhi,50
```

Expected malformed record:

```text
3,Pune,abc
```

because:

```text
qty expects IntegerType
abc cannot be parsed as INT
```

Expected reconciliation:

```text
Source rows
= 5

Returned DataFrame rows
= 4

Malformed records redirected
= 1
```

Row:

```text
4,Chennai,40,EXTRA
```

is expected to remain under the CSV behaviour being tested, with the extra token outside the supplied schema discarded.

This expectation should still be verified against the actual notebook output.

---

## 4. Lazy Evaluation

Creating the DataFrame:

```python
df = (
    spark.read
    .option("badRecordsPath", BAD_PATH)
    .schema(schema)
    .csv(CSV_PATH)
)
```

does not necessarily parse all records immediately.

The parsing work occurs when Spark executes the plan.

Example:

```python
try:
    rows = df.collect()

except Exception as e:
    ...
```

### Important

The **action that triggers parsing** must be inside the `try`.

Do not say that `spark.read.csv(...)` itself must always be inside the `try`.

---

## 5. What to Validate

Do not validate only:

```text
number of physical files created
```

Spark controls physical output layout, so one malformed record does not necessarily mean exactly one physical file.

Validate the actual data behaviour:

```text
1. Source row count
2. Returned DataFrame row count
3. Malformed row missing from returned DataFrame
4. Expected valid rows still present
5. Bad-record content captured externally
6. Failure reason / diagnostic information
```

---

## 6. Do Not Hide Path Errors

Avoid patterns such as:

```python
try:
    ...
except Exception:
    pass
```

because:

```text
Files found = 0
```

could mean:

```text
No bad records were written
```

or:

```text
The path could not be listed
Permission failed
The operation failed
```

Those are different outcomes.

Always expose the listing/access error.

---

## 7. Reconciliation

For this exercise:

```text
Source rows
      =
Returned rows
      +
Malformed rows redirected
```

Expected:

```text
5 = 4 + 1
```

This reconciliation is more important than the number of physical files created under `badRecordsPath`.

---

## 8. Production Caveats

`badRecordsPath` is a file-based mechanism.

Databricks documents important limitations:

- It is non-transactional.
- It can produce inconsistent results in some failure scenarios.
- Transient errors can be treated as failures.

Therefore it should not automatically be treated as the preferred enterprise quarantine architecture.

For Auto Loader, Databricks recommends `_corrupt_record` rather than `badRecordsPath` because potential race conditions can cause corrupted records to be missed.

Depending on the architecture, other patterns may include:

```text
_corrupt_record
_rescued_data
Auto Loader
Lakeflow expectations
explicit quarantine Delta tables
```

---

## 9. S3.20 vs S3.21

### S3.20

```text
PERMISSIVE + _corrupt_record
```

Malformed row:

```text
remains in returned DataFrame
+
raw source line available in a column
```

### S3.21

```text
PERMISSIVE + badRecordsPath
```

Malformed row:

```text
redirected to external files
+
not present in returned DataFrame
```

---

## 10. Production Flow

```text
Capture
   ↓
Inspect
   ↓
Classify
   ↓
Quarantine / remediate
   ↓
Reconcile
   ↓
Monitor
```

`badRecordsPath` is one possible capture mechanism, not the entire DQ solution.

---

## 11. Observed in Our Workspace

Not verified yet:

```text
Read succeeded              = ______

Source rows                 = 5

Returned DataFrame rows     = ______

Returned IDs                = ______

Malformed row missing       = ______

badRecordsPath accessible   = ______

Physical files found        = ______

Captured "3,Pune,abc"       = ______

Captured failure reason     = ______
```

Do not mark S3.21 complete until these values come from the actual notebook output.

---

# 12. Related Interview Questions and Answers

### Q1. What is the difference between `_corrupt_record` and `badRecordsPath`?

`_corrupt_record` keeps malformed-record information inside the returned DataFrame.

`badRecordsPath` redirects malformed-record information to external files and excludes those malformed rows from the returned DataFrame.

---

### Q2. Why should we not validate the number of bad-record files?

Because Spark controls physical file layout.

The important validation is whether the expected malformed data was captured, not whether Spark created exactly one file.

---

### Q3. Why does the action belong inside `try/except`?

Because Spark is lazy.

The parser may not process the malformed record until an action such as `collect()` executes the plan.

---

### Q4. Is `badRecordsPath` a complete DQ solution?

No.

It captures malformed input externally, but the pipeline still needs inspection, classification, reconciliation, remediation and monitoring.

---

### Q5. Is `badRecordsPath` transactional?

No.

Databricks documents the file-based mechanism as non-transactional, so it should not automatically be treated as the preferred enterprise quarantine design.

---

### Q6. Which is preferred for Auto Loader: `_corrupt_record` or `badRecordsPath`?

Databricks recommends `_corrupt_record` over `badRecordsPath` for Auto Loader because of potential race conditions with file-based bad-record capture.

---

## 13. Core Understanding

```text
Need malformed record inside DataFrame?
→ _corrupt_record

Need malformed record redirected externally?
→ badRecordsPath

Need malformed row silently removed?
→ DROPMALFORMED

Need strict ingestion failure?
→ FAILFAST

Need enterprise DQ?
→ capture + classify + quarantine + reconcile + monitor
```

**S3.21 — badRecordsPath remains open until actual workspace output is verified.**